In [ ]:
from abc import ABC, abstractmethod
import numpy as np

In [ ]:
class PolicyBuilder(ABC):
    """Classe base abstrata para construção de matrizes de política estocástica pi(a|s)."""

    def __init__(self, seed: int = None):
        self.rng = np.random.default_rng(seed)

    @abstractmethod
    def build(self, n_states: int, n_actions: int) -> np.ndarray:
        """Gera e retorna a matriz de política de formato (n_states, n_actions)."""
        pass

    @staticmethod
    def to_action_lists(pi_prob: np.ndarray) -> list[list[int]]:
        """Converte a matriz de probabilidades pi(a|s) em uma lista de ações ativas por estado.

        Retorna: [[a_0, a_1], [a_2], ...] para cada estado s.
        """
        return [np.where(row > 0)[0].tolist() for row in pi_prob]

    @staticmethod
    def to_deterministic(pi_prob: np.ndarray) -> np.ndarray:
        """Extrai a ação de maior probabilidade (greedy) para cada estado."""
        return np.argmax(pi_prob, axis=1)

In [ ]:
class RandomDeterministicPolicyBuilder(PolicyBuilder):
    """Escolhe aleatoriamente 1 ação por estado com probabilidade 1.0."""

    def build(self, n_states: int, n_actions: int) -> np.ndarray:
        policy = np.zeros((n_states, n_actions), dtype=float)
        # Escolhe um índice de ação aleatório para cada estado
        chosen_actions = self.rng.integers(0, n_actions, size=n_states)
        policy[np.arange(n_states), chosen_actions] = 1.0
        return policy

In [ ]:
class EquiprobablePolicyBuilder(PolicyBuilder):
    """Gera políticas onde o suporte de ações ativas é distribuído equiprovavelmente.

    - k_actions: int fixo, ou 'random' para sortear k em cada estado.
    """

    def __init__(self, k_actions: int | str = "random", seed: int = None):
        super().__init__(seed=seed)
        self.k_actions = k_actions

    def build(self, n_states: int, n_actions: int) -> np.ndarray:
        policy = np.zeros((n_states, n_actions), dtype=float)

        for s in range(n_states):
            # Define quantas ações serão ativas para o estado s
            if self.k_actions == "random":
                # Sorteia k entre 1 e n_actions
                k = self.rng.integers(1, n_actions + 1)
            else:
                k = min(int(self.k_actions), n_actions)

            # Sorteia k ações distintas sem reposição dentre as n_actions disponíveis
            active_actions = self.rng.choice(n_actions, size=k, replace=False)

            # Divide a probabilidade igualmente entre as ações selecionadas
            policy[s, active_actions] = 1.0 / k

        return policy

In [ ]:
# Configuração do ambiente
N_STATES = 3
N_ACTIONS = 4

In [ ]:
# 1. Política Determinística Aleatória
builder_det = RandomDeterministicPolicyBuilder(seed=42)
pi_det = builder_det.build(N_STATES, N_ACTIONS)
print("Política Determinística Aleatória:")
print(pi_det)
# Conversão elegante em 1 linha
pi_det_actions = PolicyBuilder.to_action_lists(pi_det)
print("Ações ativas por estado:")
print(pi_det_actions)

Política Determinística Aleatória:
[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]
Ações ativas por estado:
[[0], [3], [2]]


In [ ]:
# 2. Política com exatamente 3 ações equiprováveis sorteadas por estado
builder_eq3 = EquiprobablePolicyBuilder(k_actions=3, seed=42)
pi_eq3 = builder_eq3.build(N_STATES, N_ACTIONS)
print("\nPolítica com exatamente 3 ações equiprováveis sorteadas por estado:")
print(pi_eq3)
pi_eq3_actions = PolicyBuilder.to_action_lists(pi_eq3)
print("Ações ativas por estado:")
print(pi_eq3_actions)


Política com exatamente 3 ações equiprováveis sorteadas por estado:
[[0.33333333 0.         0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]]
Ações ativas por estado:
[[0, 2, 3], [0, 1, 2], [1, 2, 3]]


In [ ]:
# 3. Política com k de ações equiprováveis sorteado dinamicamente por estado
builder_eq_rand = EquiprobablePolicyBuilder(k_actions="random", seed=42)
pi_eq_rand = builder_eq_rand.build(N_STATES, N_ACTIONS)
print("\nPolítica com k de ações equiprováveis sorteado dinamicamente por estado:")
print(pi_eq_rand)
pi_eq_rand_actions = PolicyBuilder.to_action_lists(pi_eq_rand)
print("Ações ativas por estado:")
print(pi_eq_rand_actions)


Política com k de ações equiprováveis sorteado dinamicamente por estado:
[[0.         0.         0.         1.        ]
 [0.33333333 0.33333333 0.         0.33333333]
 [1.         0.         0.         0.        ]]
Ações ativas por estado:
[[3], [0, 1, 3], [0]]
